In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def get_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "data").exists():
        return cwd
    if (cwd.parent / "data").exists():
        return cwd.parent
    return cwd


PROJECT_ROOT = get_project_root()
DATA_PATH = PROJECT_ROOT / "data" / "wine" / "winequality-red.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def load_and_preprocess_wine_data(filepath: str | Path):

    df = pd.read_csv(filepath, sep=';')
    
    if df.isnull().sum().any():
        df = df.dropna() 

    X = df.drop('quality', axis=1)
    y = df['quality']

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )
    
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    print(f"Размеры выборок -> Train: {X_train_scaled.shape[0]}, Val: {X_val_scaled.shape[0]}, Test: {X_test_scaled.shape[0]}")
    
    return X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test

X_train_scaled, X_val_scaled, X_test_scaled, y_train, y_val, y_test = load_and_preprocess_wine_data(DATA_PATH)

FileNotFoundError: [Errno 2] No such file or directory: '../data/winequality-red.csv'

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

def train_and_tune_models(X_train, y_train):
    """
    Обучает 4 классификатора с подбором гиперпараметров через GridSearchCV (5-fold CV).
    Возвращает словарь с лучшими обученными моделями.
    """
    # Словарь с моделями и сетками параметров для перебора
    # Параметры можно расширять, но для универа этого хватит с головой
    models = {
        'Random Forest': {
            'model': RandomForestClassifier(random_state=42),
            'params': {
                'n_estimators': [50, 100, 200],
                'max_depth': [None, 10, 20],
                'min_samples_split': [2, 5]
            }
        },
        'Logistic Regression': {
            'model': LogisticRegression(max_iter=1000, random_state=42),
            'params': {
                'C': [0.1, 1, 10],
                'solver': ['lbfgs', 'liblinear']
            }
        },
        'SVM': {
            # probability=True обязателен, иначе потом не сможем построить ROC-кривую (AUC)
            'model': SVC(random_state=42, probability=True), 
            'params': {
                'C': [0.1, 1, 10],
                'kernel': ['linear', 'rbf']
            }
        },
        'k-NN': {
            'model': KNeighborsClassifier(),
            'params': {
                'n_neighbors': [3, 5, 7],
                'weights': ['uniform', 'distance']
            }
        }
    }

    best_estimators = {}

    # Запускаем перебор для каждой модели
    for name, config in models.items():
        print(f"Запуск тюнинга для {name}...")
        
        # cv=5 — это те самые 5 фолдов для кросс-валидации
        # n_jobs=-1 грузит все ядра твоего процессора, чтобы считалось быстрее
        grid = GridSearchCV(
            estimator=config['model'], 
            param_grid=config['params'], 
            cv=5, 
            scoring='accuracy', 
            n_jobs=-1
        )
        
        # Учим
        grid.fit(X_train, y_train)

        print(f"Лучшие параметры: {grid.best_params_}")
        print(f"Лучший скор на CV: {grid.best_score_:.4f}\n")

        # Сохраняем модель с лучшими параметрами
        best_estimators[name] = grid.best_estimator_

    return best_estimators

# Пример вызова (используя данные из предыдущего шага):
best_models = train_and_tune_models(X_train_scaled, y_train)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
from sklearn.preprocessing import label_binarize


def evaluate_and_plot(models, X_test, y_test, classes):
    """
    Прогоняет модели на тестовой выборке, печатает метрики 
    и рисует Confusion Matrix и усредненную ROC-кривую (One-vs-Rest).
    """
    # Биннаризуем метки для мультиклассового ROC AUC
    y_test_bin = label_binarize(y_test, classes=classes)
    n_classes = y_test_bin.shape[1]
    
    # Подготавливаем холсты для графиков
    fig_cm, axes_cm = plt.subplots(2, 2, figsize=(14, 12))
    axes_cm = axes_cm.flatten()
    
    plt.figure(figsize=(10, 8)) # Холст для ROC-кривых
    
    for idx, (name, model) in enumerate(models.items()):
        print(f"\n{'='*15} Оценка: {name} {'='*15}")
        
        # 1. Текстовые метрики
        y_pred = model.predict(X_test)
        # zero_division=0 чтобы скрипт не падал, если модель проигнорировала какой-то редкий класс
        print(classification_report(y_test, y_pred, zero_division=0)) 
        
        # 2. Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_cm[idx], 
                    xticklabels=classes, yticklabels=classes)
        axes_cm[idx].set_title(f'Матрица ошибок: {name}')
        axes_cm[idx].set_xlabel('Предсказанный класс')
        axes_cm[idx].set_ylabel('Истинный класс')
        
        # 3. ROC AUC (Macro Average One-vs-Rest)
        # Получаем вероятности (у k-NN, RF, LR и нашего SVM с probability=True есть predict_proba)
        y_score = model.predict_proba(X_test)
        
        # Считаем общий ROC AUC score для отчета
        auc_score = roc_auc_score(y_test, y_score, multi_class='ovr', average='macro')
        print(f"ROC AUC Score (Macro OvR): {auc_score:.4f}")
        
        # Вычисляем усредненную кривую для графика
        fpr_grid = np.linspace(0.0, 1.0, 1000)
        mean_tpr = np.zeros_like(fpr_grid)
        
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
            mean_tpr += np.interp(fpr_grid, fpr, tpr)
            
        mean_tpr /= n_classes
        
        # Рисуем
        plt.plot(fpr_grid, mean_tpr, lw=2, label=f'{name} (AUC = {auc_score:.2f})')

    # Настраиваем график матриц ошибок
    fig_cm.tight_layout()
    plt.show() # Если работаешь не в блокноте, а в скрипте, график вылезет в отдельном окне
    
    # Настраиваем финальный вид графика ROC-кривых
    plt.plot([0, 1], [0, 1], 'k--', lw=2) # Диагональная линия случайного угадывания
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (Macro Average)')
    plt.ylabel('True Positive Rate (Macro Average)')
    plt.title('Сравнение ROC-кривых (Multiclass OvR)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.savefig(RESULTS_DIR / 'confusion_matrices.png', bbox_inches='tight')
    plt.close() # Очищаем память, чтобы графики не накладывались

# Пример вызова:
# Для начала вытащим все уникальные классы из тренировочной выборки, 
# чтобы передать их в функцию
import numpy as np
unique_classes = np.unique(y_train)
evaluate_and_plot(best_models, X_test_scaled, y_test, unique_classes)